[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Query Parameters &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API and `station_weather`.
Run it first.


In [1]:
import importlib
import sys
import urllib.request
from datetime import date
from pathlib import Path
from urllib.parse import quote, urlencode

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()


def station_weather(latitude, longitude, unit=None):
    """Three daily mean temperatures for a place, from Open-Meteo, in Celsius unless a unit is given."""
    return requests.get(OPEN_METEO, timeout=30, params={
        "latitude": latitude, "longitude": longitude,
        "start_date": date(2025, 1, 15).isoformat(), "end_date": date(2025, 1, 17).isoformat(),
        "daily": "temperature_2m_mean", "models": "era5", "temperature_unit": unit})


print("ready:", BASE)
print("Open-Meteo:", OPEN_METEO)


ready: http://127.0.0.1:8765
Open-Meteo: https://archive-api.open-meteo.com/v1/archive


**1.** Two values that need encoding.


In [2]:
response = requests.get(f"{BASE}/echo", params={"station": "Tromsø", "note": "cold & dark"}, timeout=10)

print(response.url)
print(response.json()["args"])


http://127.0.0.1:8765/echo?station=Troms%C3%B8&note=cold+%26+dark
{'station': ['Tromsø'], 'note': ['cold & dark']}


`ø` traveled as two escapes, `%C3%B8`, one for each byte of its UTF-8 encoding, and `&` as `%26`,
so the server received both values whole.


**2.** A query built by hand, both ways.


In [3]:
query = {"q": "50% chance of snow"}

print(urlencode(query))
print(urlencode(query, quote_via=quote))


q=50%25+chance+of+snow
q=50%25%20chance%20of%20snow


The `%` is `%25` in both, so the server can tell it from the start of an escape. Only the spaces
differ.


**3.** A list, both ways.


In [4]:
ids = ["oslo", "bergen", "tromso"]

for params in [{"id": ids}, {"ids": ",".join(ids)}]:
    print(requests.get(f"{BASE}/echo", params=params, timeout=10).json()["args"])


{'id': ['oslo', 'bergen', 'tromso']}
{'ids': ['oslo,bergen,tromso']}


Three values for one name, then one value holding three ids. An API reads the form its
documentation describes, and may read the other as something else, so send the documented one.


**4.** A Python flag and a documented flag.


In [5]:
response = requests.get(f"{BASE}/echo", params={"python": True, "documented": "true"}, timeout=10)

print(response.json()["query"])


python=True&documented=true


`str(True)` is `True`, with a capital letter. Only the string gives the lowercase `true` that most
APIs document for a flag.


**5.** A unit that is not `None`.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [6]:
response = station_weather(69.65, 18.96, "fahrenheit")

print(response.url)
print(response.json()["daily_units"]["temperature_2m_mean"], response.json()["daily"]["temperature_2m_mean"])


https://archive-api.open-meteo.com/v1/archive?latitude=69.65&longitude=18.96&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5&temperature_unit=fahrenheit
°F [40.3, 43.5, 44.8]


`temperature_unit` is in the URL because `unit` was not `None`, and the `°F` confirms that
Open-Meteo used it.


**6.** A question mark, encoded and not.


In [7]:
for path_value in [quote("svalbard?", safe=""), "svalbard?"]:
    response = requests.get(f"{BASE}/stations/{path_value}", timeout=10)
    body = response.json()
    print(f"{path_value:<12}", response.status_code, body.get("name") or body["error"])


svalbard%3F  404 no station with id 'svalbard?'
svalbard?    200 Svalbard


Encoded, the id stayed one segment, and the practice API looked for a station called `svalbard?`.
Written as it is, the `?` began an empty query, so the path was `/stations/svalbard`, and the
practice API found Svalbard.


---

&#8592; **Back to:** [Query Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/05-query-parameters.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
